In [2]:
import os

BASE_DIR = r"C:\Users\Bachirou\Project\ANPER_PROJECT"
LOGO_PATH = os.path.join(BASE_DIR, "assets", "seec_icon.png")

print(f"Expected path: {LOGO_PATH}")
print(f"Folder exists: {os.path.exists(os.path.dirname(LOGO_PATH))}")
print(f"File exists: {os.path.exists(LOGO_PATH)}")

# List files in assets folder
assets_dir = os.path.join(BASE_DIR, "assets")
if os.path.exists(assets_dir):
    print(f"Files in assets: {os.listdir(assets_dir)}")
else:
    print("Assets folder does not exist - creating it...")
    os.makedirs(assets_dir)

Expected path: C:\Users\Bachirou\Project\ANPER_PROJECT\assets\seec_icon.png
Folder exists: True
File exists: False
Files in assets: ['seec_icon.jpeg', 'seec_icon1.png', 'seec_icon2.jpeg']


In [5]:
import pandas as pd
import os

# ============================================================================
# Set the path manually
# ============================================================================
BASE_DIR = r"C:\Users\Bachirou\Project\ANPER_PROJECT"
DATA_PATH = os.path.join(BASE_DIR, "data", "vida_clean.parquet")

print("=" * 70)
print("DATA PATH")
print("=" * 70)
print(f"BASE_DIR: {BASE_DIR}")
print(f"DATA_PATH: {DATA_PATH}")
print(f"File exists: {os.path.exists(DATA_PATH)}")
print()

# ============================================================================
# 1. Load the raw data
# ============================================================================
print("=" * 70)
print("STEP 1: Load raw data from parquet")
print("=" * 70)

if not os.path.exists(DATA_PATH):
    print(f"❌ ERROR: File not found at {DATA_PATH}")
else:
    print(f"✅ Loading from: {DATA_PATH}")
    df = pd.read_parquet(DATA_PATH)
    print(f"✅ Successfully loaded!")
    print(f"Shape: {df.shape}")
    print(f"\nAll columns ({len(df.columns)} total):\n")
    for i, col in enumerate(df.columns, 1):
        print(f"  {i}. {col}")

    # ============================================================================
    # 2. Check nightlight column specifically
    # ============================================================================
    print("\n" + "=" * 70)
    print("STEP 2: Check nightlight column")
    print("=" * 70)

    nightlight_col = "Éclairage nocturne [%]"
    print(f"Nightlight column name: '{nightlight_col}'")
    print(f"Column exists in df: {nightlight_col in df.columns}")

    if nightlight_col in df.columns:
        print(f"\n✅ Column found!")
        print(f"Data type: {df[nightlight_col].dtype}")
        print(f"Non-null count: {df[nightlight_col].notna().sum()}/{len(df)}")
        print(f"Null count: {df[nightlight_col].isna().sum()}")
        print(f"Min value: {df[nightlight_col].min()}")
        print(f"Max value: {df[nightlight_col].max()}")
        print(f"Mean value: {df[nightlight_col].mean():.2f}")
        print(f"\nFirst 10 values:")
        print(df[nightlight_col].head(10).tolist())
        print(f"\nSample rows with nightlight:")
        name_col = "Nom"
        print(df[[name_col, nightlight_col]].head(10))
    else:
        print(f"\n❌ Column NOT found in dataframe!")
        print(f"Available columns with 'Éclairage' or 'night':")
        matching = [c for c in df.columns if 'clair' in c.lower() or 'night' in c.lower()]
        if matching:
            print(matching)
        else:
            print("  (none found)")

    # ============================================================================
    # 3. Check if values are being read correctly
    # ============================================================================
    print("\n" + "=" * 70)
    print("STEP 3: Check data types and values")
    print("=" * 70)

    if nightlight_col in df.columns:
        # Convert to numeric
        nightlight_numeric = pd.to_numeric(df[nightlight_col], errors="coerce")
        print(f"After converting to numeric:")
        print(f"  Non-null numeric: {nightlight_numeric.notna().sum()}")
        print(f"  Min: {nightlight_numeric.min()}")
        print(f"  Max: {nightlight_numeric.max()}")
        print(f"\nSample converted values:")
        print(df[[name_col, nightlight_col]].head(10))

    # ============================================================================
    # 4. Simulate filtering (like filters.py does)
    # ============================================================================
    print("\n" + "=" * 70)
    print("STEP 4: Simulate filtering process")
    print("=" * 70)

    filtered = df.copy()

    # Apply some basic filters (similar to what filters.py does)
    region_col = "Région"
    if region_col in filtered.columns:
        filtered = filtered[filtered[region_col].notna()]

    if "Score_Viabilité" in filtered.columns:
        filtered = filtered[filtered["Score_Viabilité"].between(0, 100)]

    print(f"Filtered shape: {filtered.shape}")
    print(f"Nightlight in filtered: {nightlight_col in filtered.columns}")

    if nightlight_col in filtered.columns:
        print(f"\nAfter filtering:")
        print(f"  Non-null count: {filtered[nightlight_col].notna().sum()}/{len(filtered)}")
        print(f"  Min: {filtered[nightlight_col].min()}")
        print(f"  Max: {filtered[nightlight_col].max()}")
        print(f"\nFirst 5 nightlight values in filtered:")
        print(filtered[nightlight_col].head().tolist())

        # Check for zero values
        zero_count = (filtered[nightlight_col] == 0).sum()
        print(f"\nRows with nightlight == 0: {zero_count}/{len(filtered)}")

        if zero_count > 0:
            print(f"Sample rows with 0 nightlight:")
            print(filtered[filtered[nightlight_col] == 0][[name_col, nightlight_col]].head())

    # ============================================================================
    # 5. Check specific location from your debug output
    # ============================================================================
    print("\n" + "=" * 70)
    print("STEP 5: Check specific locations from your debug output")
    print("=" * 70)

    locations = ["Birni N'Konni", "Éini Goungou", "Aguié"]

    for loc in locations:
        matching = df[df[name_col] == loc]
        if len(matching) > 0:
            row = matching.iloc[0]
            print(f"\n✅ {loc}:")
            print(f"  Nightlight: {row.get(nightlight_col, 'NOT FOUND')}")
        else:
            print(f"\n❌ {loc}: NOT FOUND in dataframe")

    # ============================================================================
    # 6. Check the actual data in your vida_clean.parquet
    # ============================================================================
    print("\n" + "=" * 70)
    print("STEP 6: Direct parquet inspection")
    print("=" * 70)

    try:
        import pyarrow.parquet as pq

        parquet_file = pq.read_table(DATA_PATH)
        print(f"Parquet column names ({len(parquet_file.column_names)} total):")
        for i, col in enumerate(parquet_file.column_names, 1):
            print(f"  {i}. {col}")

        if nightlight_col in parquet_file.column_names:
            nightlight_data = parquet_file[nightlight_col].to_pandas()
            print(f"\n✅ Nightlight column from parquet:")
            print(f"  Type: {nightlight_data.dtype}")
            print(f"  Non-null: {nightlight_data.notna().sum()}/{len(nightlight_data)}")
            print(f"  First 10: {nightlight_data.head(10).tolist()}")
        else:
            print(f"\n❌ Nightlight NOT in parquet columns")
    except Exception as e:
        print(f"Error reading parquet: {e}")

    print("\n" + "=" * 70)
    print("END OF DIAGNOSTIC")
    print("=" * 70)

DATA PATH
BASE_DIR: C:\Users\Bachirou\Project\ANPER_PROJECT
DATA_PATH: C:\Users\Bachirou\Project\ANPER_PROJECT\data\vida_clean.parquet
File exists: True

STEP 1: Load raw data from parquet
✅ Loading from: C:\Users\Bachirou\Project\ANPER_PROJECT\data\vida_clean.parquet
✅ Successfully loaded!
Shape: (26309, 38)

All columns (38 total):

  1. Latitude
  2. Longitude
  3. Nombre estimé de connexions
  4. Demande énergétique estimée [kWh/day]
  5. Demande moyenne par connexion [kWh/day]
  6. Production PV potentielle [kWh/kWp]
  7. Distance au réseau existant [km]
  8. Distance au réseau planifié [km]
  9. Éclairage nocturne [%]
  10. Distance à la lumière nocturne [km]
  11. Nom
  12. Région
  13. Départment
  14. Densité des bâtiments [%]
  15. Superficie du site [km2]
  16. Nombre de bâtiments
  17. Bâtiments grands
  18. Bâtiments moyens
  19. Bâtiments petits
  20. Structures très petites
  21. Population
  22. Distance à la source d'eau [km]
  23. Établissement d'enseignement
  24. Ce

In [6]:
# Add this to find where 97.2 comes from
import pandas as pd

BASE_DIR = r"C:\Users\Bachirou\Project\ANPER_PROJECT"
DATA_PATH = os.path.join(BASE_DIR, "data", "vida_clean.parquet")

df = pd.read_parquet(DATA_PATH)

# Find Birni N'Konni
birni = df[df['Nom'] == "Birni N'Konni"]
print("Birni N'Konni row:")
print(birni.to_string())

# Search for 97.2 in the dataset
for col in df.columns:
    try:
        if (df[col] == 97.2).any():
            print(f"\n97.2 found in column: {col}")
            print(df[df[col] == 97.2][['Nom', col]])
    except:
        pass

Birni N'Konni row:
        Latitude  Longitude  Nombre estimé de connexions  Demande énergétique estimée [kWh/day]  Demande moyenne par connexion [kWh/day]  Production PV potentielle [kWh/kWp]  Distance au réseau existant [km]  Distance au réseau planifié [km]  Éclairage nocturne [%]  Distance à la lumière nocturne [km]            Nom  Région     Départment  Densité des bâtiments [%]  Superficie du site [km2]  Nombre de bâtiments  Bâtiments grands  Bâtiments moyens  Bâtiments petits  Structures très petites  Population  Distance à la source d'eau [km] Établissement d'enseignement Centres de santé  Indice de richesse relative Accès routier principal?  Distance à la route principale [km]  Distance au centre urbain [km] Centre urbain le plus proche Décès dans un rayon de 50 km (batailles:émeutes:violence contre les civils:explosions) Décès dans un rayon de 25 km (batailles:émeutes:violence contre les civils:explosions)  Incidents dans un rayon de 50 km Risque de sécurité                  